# VOLE-Field — durable generative-state reuse

**One proposition, one experiment.** A tiny recurrent video model (ConvLSTM) watches a
synthetic scene and accumulates recurrent state. That state is persisted through
[EntropyFS](https://crates.io/crates/entropyfs) by one process, which then exits. A
*different* process restores the state and generates related futures without replaying the
observed history.

Everything here is native Rust on CPU. **Runtime → Run all.** No GPU, no API key, no
account, no download. The whole notebook is four steps: checkout, build, run, look.

**Notebook revision 7.** Cell 1 prints this revision and checks it against `main`, so
a stale copy announces itself. Colab loads a notebook once and never re-fetches it, so
an old tab can run old cell source against a freshly cloned repository.


In [ ]:
# Cell 1 -- environment and the exact commit under test.
REPO_URL = "https://github.com/infinityabundance/vole-field"
COMMIT = "main"
WORKDIR = "/content/vole-field"

# Revision of *this* notebook. Bump it whenever the notebook changes -- and bump the
# matching number in the markdown cell at the top, which is what a reader sees before
# anything runs. Cell 1 compares this number with the notebook in the checkout.
NOTEBOOK_REVISION = 7

import json, os, re, shutil, subprocess, threading, time

def sh(cmd, cwd=None, heartbeat=30):
    """Run a command, streaming its output as it arrives.

    Deliberately not subprocess.run(): Colab hands the child a pipe rather than a
    terminal, and something that block-buffers can look hung for minutes while it is
    working perfectly hard. Reading line by line and flushing each line means the
    notebook never looks stuck for no reason, and the heartbeat covers the long
    stretches where the tool genuinely says nothing.
    """
    banner = "$ " + " ".join(cmd)
    LOG.append(banner)
    print("\n" + banner, flush=True)
    started = time.time()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    done = threading.Event()

    def beat():
        while not done.wait(heartbeat):
            print("    ... still working, {:.0f}s elapsed".format(time.time() - started),
                  flush=True)

    threading.Thread(target=beat, daemon=True).start()
    try:
        for line in proc.stdout:
            LOG.append(line.rstrip("\n"))
            print(line.rstrip("\n"), flush=True)
    finally:
        done.set()
    code = proc.wait()
    elapsed = "    exit {} after {:.1f}s".format(code, time.time() - started)
    LOG.append(elapsed)
    print(elapsed, flush=True)
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code


# Every line the commands below print, kept so the downloadable archive can carry the
# narrative of how the numbers were produced and not only the numbers themselves.
LOG = []

# Rust toolchain. rust-toolchain.toml in the checkout pins the exact compiler, so
# rustup installs that version on first use.
if shutil.which("cargo") is None:
    sh(["bash", "-c",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
        "| sh -s -- -y --profile minimal --default-toolchain none"])
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Checkout. If you only have a local tree, copy it to WORKDIR first and set REPO_URL
# to a file:// URL, or skip the clone by pre-creating WORKDIR.
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    sh(["git", "-C", WORKDIR, "fetch", "--quiet", "origin"])
elif not os.path.isdir(WORKDIR):
    sh(["git", "clone", "--quiet", REPO_URL, WORKDIR])
sh(["git", "-C", WORKDIR, "checkout", "--quiet", COMMIT])

rev = subprocess.run(["git", "-C", WORKDIR, "rev-parse", "HEAD"],
                     capture_output=True, text=True)
print("\nrepository :", REPO_URL)
if rev.returncode == 0:
    print("commit     :", rev.stdout.strip())
    print(subprocess.run(["git", "-C", WORKDIR, "log", "-1", "--format=%H%n%ad%n%s"],
                         capture_output=True, text=True).stdout)
else:
    print("commit     : (this checkout has no commit yet)")


# ---- is this copy the latest? -------------------------------------------------
# Placed after the checkout but before the first compiler call, because a stale
# notebook's most likely symptom is a failure right there -- which is exactly when
# this needs to have been said.
#
# It reads the notebook from the *checkout* rather than from raw.githubusercontent.com
# on purpose. The raw CDN serves a copy for minutes after a push, and a staleness check
# that can itself be stale is worse than no check. The checkout arrived over the git
# protocol, which does not have that problem.
def notebook_revision(nb):
    """The NOTEBOOK_REVISION a notebook declares, or None if it declares none."""
    text = "\n".join("".join(c.get("source", [])) for c in nb.get("cells", []))
    m = re.search(r"^NOTEBOOK_REVISION\s*=\s*(\d+)\s*$", text, re.M)
    return int(m.group(1)) if m else None


def check_notebook_revision():
    path = os.path.join(WORKDIR, "colab", "vole_field_demo.ipynb")
    print("notebook   : revision {} (this copy)".format(NOTEBOOK_REVISION))
    try:
        with open(path) as f:
            upstream = notebook_revision(json.load(f))
    except Exception as e:
        print("             could not read {}: {}".format(path, e))
        return
    if upstream is None:
        print("             {} declares no revision".format(path))
    elif upstream == NOTEBOOK_REVISION:
        print("             revision {} in the checkout -- this copy is current\n"
              .format(upstream))
    elif upstream > NOTEBOOK_REVISION:
        print("             revision {} in the checkout -- THIS COPY IS OLD\n"
              .format(upstream))
        print("  *** STALE NOTEBOOK: this tab is running an old copy of the cell source. ***")
        print("  Colab loads the .ipynb once and does not re-fetch it on 'Run all', and the")
        print("  raw CDN can serve a stale copy for minutes after a push. Close this tab,")
        print("  reopen the notebook, and run again.\n")
    else:
        print("             revision {} in the checkout -- this copy is NEWER\n"
              .format(upstream))
        print("  Ahead of the checkout, which is expected if you are editing the notebook")
        print("  locally or if the change has not been pushed.\n")


check_notebook_revision()

# The pinned compiler comes from rust-toolchain.toml, and rustup only consults that
# file for the *current* directory. rustup was installed above with
# --default-toolchain none, so invoked from anywhere else these fail with
# "no default toolchain configured" and exit 1. cwd=WORKDIR is what selects the
# toolchain -- and this call is also what downloads it on first use.
print("toolchain  :", [l for l in open(os.path.join(WORKDIR, "rust-toolchain.toml"))
                        if l.startswith("channel")][0].strip())
sh(["rustc", "-V"], cwd=WORKDIR)
sh(["cargo", "-V"], cwd=WORKDIR)

In [ ]:
# Cell 2 -- build. --locked refuses to move any dependency version.
#
# This is the slow step, for a boring reason: a cold release build of the entire
# dependency graph on a free Colab CPU. Expect roughly 10-20 minutes. The README's
# "~30 seconds" is a 16-core number and does not transfer -- much of the build is
# per-crate, and several crates compile C or assembly (gemm, blake3), which is serial.
#
# Output streams, so a quiet stretch with a heartbeat means a big crate is compiling,
# not that anything has hung.
packages = sum(1 for line in open(os.path.join(WORKDIR, "Cargo.lock"))
               if line.startswith("[[package]]"))
print("dependencies to build : {}".format(packages))
print("cpus visible to python: {}".format(os.cpu_count()))
print("this cell takes minutes on a free Colab CPU -- it is the slow one")
sh(["cargo", "build", "--release", "--locked"], cwd=WORKDIR)
print("\nbuild complete; cell 3 runs the experiment")

In [ ]:
# Cell 3 -- run. This orchestrates the whole experiment as child processes:
#           producer (earn + persist + exit), baseline (replay + generate),
#           raw checkpoint, and the EntropyFS restore path. The killer table is here.
#
# Streamed as well as captured: the experiment prints its own progress (producer,
# identity, timing, reuse, negative case) and watching that arrive is the point.
lines = []
proc = subprocess.Popen(["cargo", "run", "--release", "--locked"], cwd=WORKDIR,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    lines.append(line)
    LOG.append(line.rstrip("\n"))
    print(line.rstrip("\n"), flush=True)
code = proc.wait()
output = "".join(lines)
assert code == 0, "the run reported FAIL (exit {})".format(code)

In [ ]:
# Cell 4 -- look at the evidence. Python only draws what Rust produced: no inference,
#           no persistence, no benchmarking, no output comparison.
import csv
from PIL import Image
import matplotlib.pyplot as plt

RUN = os.path.join(WORKDIR, "run")

montage = Image.open(os.path.join(RUN, "branches.ppm"))
fig, ax = plt.subplots(figsize=(10.5, 11.8))
ax.imshow(montage)
ax.axis("off")
ax.set_title(
    "One earned state, four related futures\n"
    "row 1: the observed context tail\n"
    "rows 2-5: the model's continue / turn-left / turn-right / accelerate,\n"
    "          generated after the producing process had already exited\n"
    "rows 6-9: the same four branches, generated by the scene itself (ground truth)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

rows = list(csv.DictReader(open(os.path.join(RUN, "reuse.csv"))))
n = [int(r["N"]) for r in rows]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(n, [float(r["baseline_ms"]) for r in rows], "o-", label="baseline (replay history)")
ax.plot(n, [float(r["raw_checkpoint_ms"]) for r in rows], "s--", label="raw tensor checkpoint")
ax.plot(n, [float(r["vole_ms"]) for r in rows], "^-", label="VOLE / EntropyFS restore")
ax.set_xscale("log", base=2)
ax.set_xticks(n)
ax.set_xticklabels([str(v) for v in n])
ax.set_xlabel("N related future requests (each in its own fresh process)")
ax.set_ylabel("cumulative wall time (ms)")
ax.set_title("Cumulative cost, including the one-time state persistence")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(open(os.path.join(RUN, "reuse.csv")).read())

In [ ]:
# Cell 5 -- take everything away: one zip, holding the results and the log of how they
#           were produced.
#
# Nothing is written into run/ here, so that directory stays exactly what the Rust
# program emitted and every file in it is attributable to the binary. The notebook's
# own log is added beside it, at the top level of the archive.
import os, zipfile

RUN = os.path.join(WORKDIR, "run")
ZIP = "/content/vole-field-results.zip"
LOGFILE = "/content/vole-field-notebook-log.txt"

with open(LOGFILE, "w") as f:
    f.write("vole-field -- notebook log\n")
    f.write("notebook revision : {}\n".format(NOTEBOOK_REVISION))
    f.write("repository        : {}\n".format(REPO_URL))
    f.write("requested ref     : {}\n".format(COMMIT))
    f.write("=" * 78 + "\n\n")
    f.write("\n".join(LOG))
    f.write("\n")

count = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root, _, names in os.walk(RUN):
        for name in sorted(names):
            path = os.path.join(root, name)
            z.write(path, os.path.relpath(path, os.path.dirname(RUN)))
            count += 1
    z.write(LOGFILE, os.path.basename(LOGFILE))

size = os.path.getsize(ZIP)
print("results files in archive : {}".format(count))
print("notebook log lines       : {}".format(len(LOG)))
print("archive                  : {} ({:.2f} MiB)".format(ZIP, size / 1048576.0))
print()
print("contents:")
for name in sorted(z.namelist())[:12]:
    print("  " + name)
if count + 1 > 12:
    print("  ... and {} more".format(count + 1 - 12))

try:
    from google.colab import files
except ImportError:
    print("\nNot a Colab runtime; the archive is at {}".format(ZIP))
else:
    print("\nYour download should start now. The same file is also in Colab's file")
    print("browser (the folder icon in the left pane), under /content.")
    try:
        files.download(ZIP)
    except Exception as e:
        print("Colab's download call failed ({}). Use the file browser instead.".format(e))

### Reading the result

* The restored state is **byte-identical** to the state the producer earned, and every
  restored output is **byte-identical** to the from-scratch replay output. That is the
  correctness gate, and it is what makes the timing comparison meaningful.
* The restored path executes **fewer recurrent steps** — that is the load-bearing evidence,
  and it does not depend on the machine being quiet.
* Wall-clock shows whether the saved work was visible on *this* VM. If VOLE is slower than
  the raw checkpoint, or if there is no break-even inside the tested range, the run still
  **passes**: performance is reported, never gatekept.
* The model itself is **not** a good video model, and its control response is sub-pixel.
  That is why the montage shows the scene's own branches beside the model's: you can see
  both what the controls do and what the model makes of them. `cargo run --release -- eval`
  reports the model's quality numerically. Nothing in the claim depends on it.

The full evidence is in `run/results.json`; each child's own report is in `run/reports/`.
`README.md` lists precisely what this experiment does **not** prove.